# AnimalCLEF 2026 — Fine-tune MIEWID v3

Minimal notebook: prepare AnimalCLEF metadata as MIEW-ID CSVs, then run **two-stage fine-tuning** with `wbia_miew_id.train.Trainer` (head-only warmup + full model).


In [ ]:
# Install dependencies
!pip install -q git+https://github.com/WildMeOrg/wbia-plugin-miew-id.git
!pip install -q "transformers==4.45.2" timm albumentations
!pip install -q wildlife-datasets


In [ ]:
import os, json, shutil
import numpy as np
import pandas as pd
import torch
from transformers import AutoModel

from wbia_miew_id.etl import preprocess_data
from wbia_miew_id.helpers import get_config, formatted_string_to_yaml
from wbia_miew_id.train import Trainer

import warnings
warnings.filterwarnings("ignore")

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {DEVICE}")


In [ ]:
# === Paths (edit if your dataset layout differs) ===
INPUT_ROOT  = "/kaggle/input/datasets/seifelkerdany/animalclef26"
COMP_META   = "/kaggle/input/competitions/animal-clef-2026/metadata.csv"  # optional dates for time split
WORK_DIR    = "/kaggle/working"

IMAGE_SIZE  = (440, 440)
MODEL_TAG   = "conservationxlabs/miewid-msv3"

SPECIES_MAP = {
    "LynxID2025":          "lynx",
    "SalamanderID2025":    "salamander",
    "SeaTurtleID2022":     "sea_turtle",
    "TexasHornedLizards":  "horned_lizard",
}

TRAIN_BATCH_SIZE = 8
VALID_BATCH_SIZE = 8
CONFIG_TEST_BATCH_SIZE = 8   # YAML `test.batch_size` for Trainer
DATALOADER_NUM_WORKERS = 4


## 1 — Load metadata and build train/val CSVs

Produce `train.csv` / `val.csv` with columns `file_name, name, species, x, y, w, h, theta` for the MIEW-ID pipeline.


In [ ]:
metadata = pd.read_csv(os.path.join(INPUT_ROOT, "metadata.csv"))
print(f"Total images : {len(metadata)}")
print(f"Datasets     : {metadata['dataset'].unique().tolist()}")

# Optional: merge capture dates from competition metadata (enables time-based open-set split)
try:
    comp_meta = pd.read_csv(COMP_META)
    metadata = metadata.merge(comp_meta[["path", "date"]], on="path", how="left")
    n_dates = metadata["date"].notna().sum()
    print(f"Merged dates : {n_dates}/{len(metadata)} images have a date")
except Exception as e:
    metadata["date"] = pd.NaT
    print(f"Could not merge dates ({e}); will use identity-stratified split")

labeled = metadata[metadata["identity"].notna()].copy().reset_index(drop=True)
print(f"\nLabeled images      : {len(labeled)}")
print(f"Unique identities   : {labeled['identity'].nunique()}")
print(f"Unique datasets     : {labeled['dataset'].nunique()}")

labeled["species"] = labeled["dataset"].map(SPECIES_MAP)
labeled["file_name"] = labeled["path"].apply(lambda p: os.path.join(INPUT_ROOT, p))

# BBox + angle (crop disabled in config; placeholders)
labeled["x"] = labeled["y"] = 0
labeled["w"] = labeled["h"] = 100
labeled["theta"] = 0.0
labeled["name"] = labeled["identity"]

display(labeled.head(3))


In [ ]:
# Train / val: prefer time-proportioned open-set split; else stratify by identity per species
if "date" in labeled.columns and labeled["date"].notna().any():
    date_col = pd.to_datetime(labeled["date"], errors="coerce")
    min_d, max_d = date_col.dropna().min(), date_col.dropna().max()
    missing = date_col.isna()
    n_missing = missing.sum()
    if n_missing > 0 and min_d is not pd.NaT:
        rng = np.random.default_rng(42)
        rand_days = rng.integers(0, max(1, (max_d - min_d).days + 1), size=n_missing)
        date_col.loc[missing] = min_d + pd.to_timedelta(rand_days, unit="D")
    labeled["date"] = date_col
    print(f"Date range: {min_d} -> {max_d}  (filled {n_missing} missing dates)")

split_ok = False
try:
    from wildlife_datasets.splits import TimeProportionOpenSetSplit

    splitter = TimeProportionOpenSetSplit(ratio_train=0.85, n_class_test=50)
    idx_train, idx_val = splitter.split(labeled)[0]
    df_train = labeled.loc[idx_train].copy().reset_index(drop=True)
    df_val = labeled.loc[idx_val].copy().reset_index(drop=True)
    split_ok = True
    print("Time-based open-set split succeeded")
except Exception as e:
    print(f"Time-based split failed ({e})")

if not split_ok:
    print("Using identity-stratified split (80/20 per species)")
    rng = np.random.default_rng(42)
    train_ids, val_ids = [], []
    for _, grp in labeled.groupby("species"):
        ids = grp["identity"].unique()
        rng.shuffle(ids)
        k = int(len(ids) * 0.80)
        train_ids.extend(ids[:k])
        val_ids.extend(ids[k:])
    df_train = labeled[labeled["identity"].isin(train_ids)].reset_index(drop=True)
    df_val = labeled[labeled["identity"].isin(val_ids)].reset_index(drop=True)

print(f"\nTrain : {len(df_train)} images, {df_train['identity'].nunique()} identities")
print(f"Val   : {len(df_val)} images, {df_val['identity'].nunique()} identities")
for sp in sorted(SPECIES_MAP.values()):
    nt = df_train.loc[df_train["species"] == sp, "identity"].nunique()
    nv = df_val.loc[df_val["species"] == sp, "identity"].nunique()
    print(f"  {sp:20s}  train {nt:>4d} ids   val {nv:>4d} ids")


In [ ]:
train_csv = os.path.join(WORK_DIR, "train.csv")
val_csv = os.path.join(WORK_DIR, "val.csv")

csv_cols = ["file_name", "name", "species", "x", "y", "w", "h", "theta"]
df_train[csv_cols].to_csv(train_csv, index=False)
df_val[csv_cols].to_csv(val_csv, index=False)

print(f"Saved  {train_csv}  ({len(df_train)} rows)")
print(f"Saved  {val_csv}    ({len(df_val)} rows)")


## 2 — Fine-tune MIEWID v3

**Two stages** inside `Trainer.run(finetune=True)`:
warmup on the classification head (backbone frozen), then train the full network for `engine.epochs` in the YAML (`15` below).


In [ ]:
# Match Trainer filtering so ArcFace head size equals number of train identities
_df_tr = preprocess_data(
    train_csv,
    name_keys=["name"],
    convert_names_to_ids=True,
    viewpoint_list=None,
    n_filter_min=3,
    n_subsample_max=None,
    use_full_image_path=True,
    images_dir=None,
)
N_TRAIN_CLASSES = int(_df_tr["name"].nunique())
print(f"model_params.n_classes = {N_TRAIN_CLASSES} (after n_filter_min=3)")

config_yaml = f"""
checkpoint_dir: {WORK_DIR}/runs
comment: MIEWIDv3 fine-tune on AnimalCLEF
data:
  crop_bbox: false
  image_size:
  - {IMAGE_SIZE[0]}
  - {IMAGE_SIZE[1]}
  images_dir: null
  name_keys:
  - name
  preprocess_images:
    apply: false
    force_apply: false
    preprocessed_dir: null
  test:
    anno_path: {val_csv}
    checkpoint_path: null
    eval_groups: []
    n_filter_min: 2
    n_subsample_max: null
  train:
    anno_path: {train_csv}
    n_filter_min: 3
    n_subsample_max: null
  use_full_image_path: true
  val:
    anno_path: {val_csv}
    n_filter_min: 2
    n_subsample_max: null
  viewpoint_list: null
engine:
  device: cuda
  epochs: 15
  num_workers: {DATALOADER_NUM_WORKERS}
  seed: 42
  train_batch_size: {TRAIN_BATCH_SIZE}
  use_swa: false
  use_wandb: false
  valid_batch_size: {VALID_BATCH_SIZE}
exp_name: miewid-v3-animalclef
model_params:
  dropout: 0
  fc_dim: 2048
  k: 3
  loss_module: arcface_subcenter_dynamic
  ls_eps: 0
  margin: 0.6
  model_name: efficientnetv2_rw_m
  n_classes: {N_TRAIN_CLASSES}
  pretrained: true
  s: 64.0
  theta_zero: 0.785
  use_fc: false
project_name: miewid-finetune
scheduler_params:
  lr_decay: 0.8
  lr_max: 0.0008
  lr_min: 1.0e-06
  lr_ramp_ep: 5
  lr_start: 5.0e-05
  lr_sus_ep: 0
swa_params:
  swa_lr: 0.03
  swa_start: 24
test:
  batch_size: {CONFIG_TEST_BATCH_SIZE}
  fliplr: false
  fliplr_view: []
"""

config_path = os.path.join(WORK_DIR, "config.yaml")
formatted_string_to_yaml(config_yaml, config_path)
config = get_config(config_path)
print("Config loaded OK")


In [ ]:
model = AutoModel.from_pretrained(MODEL_TAG, trust_remote_code=True)
model.final_in_features = 2152

trainer = Trainer(config, model)
best_score = trainer.run(finetune=True)

print(f"Fine-tuning complete. Best validation score: {best_score:.6f}")


## 3 — Copy checkpoint and config

`Trainer` writes weights under `{WORK_DIR}/runs/miewid-finetune/miewid-v3-animalclef/model_best.bin`. Copies artifacts to a flat folder.


In [ ]:
checkpoint_dir = f"{WORK_DIR}/runs/miewid-finetune/miewid-v3-animalclef"
best_model_path = os.path.join(checkpoint_dir, "model_best.bin")
assert os.path.isfile(best_model_path), f"Missing checkpoint: {best_model_path}"

out_dir = os.path.join(WORK_DIR, "finetuned_miewid_v3")
os.makedirs(out_dir, exist_ok=True)

shutil.copy2(best_model_path, os.path.join(out_dir, "model_best.bin"))
shutil.copy2(config_path, os.path.join(out_dir, "config.yaml"))

summary = {
    "best_val_score": float(best_score),
    "checkpoint_source": checkpoint_dir,
}
with open(os.path.join(out_dir, "training_summary.json"), "w") as f:
    json.dump(summary, f, indent=2)

print(f"Saved under {out_dir}/")
print("  model_best.bin, config.yaml, training_summary.json")
